# Guía de funciones por cada punto del enunciado

Casi todo se resuelve con **métodos de pandas** (`Series`/`DataFrame`). Los `max()`/`min()` de Python *funcionan* sobre una Serie, pero pandas tiene equivalentes más rápidos y que devuelven `NaN` si hay nulos, así que en la práctica **usa siempre pandas** y reserva Python builtins para cosas sueltas (formatear strings, etc.). Seaborn solo lo necesitas para **visualizar outliers**.

Te dejo cada punto con la(s) función(es) exactas y un mini ejemplo sobre tu dataset.

---

## 1. Identificar valores únicos de la columna

```python
data_csv['MSZoning'].unique()          # array con los valores distintos
data_csv['MSZoning'].nunique()         # cuántos hay
data_csv['MSZoning'].value_counts()    # frecuencia de cada uno (muy útil)
data_csv['MSZoning'].value_counts(normalize=True)  # en proporción
```

- `unique()` → lista cruda (útil para ver los valores).
- `nunique()` → un número (útil para decidir si recodificar: si hay >20 categorías distintas, cuidado).
- `value_counts()` → el más informativo. Para numéricas también sirve, pero suele ser mejor `describe()`.

---

## 2. Determinar el tipo (categórica sin/con orden, discreta, continua)

No hay una función mágica: **pandas te da el `dtype`, tú decides la clasificación**. Ayuda:

```python
data_csv['MSZoning'].dtype        # object/str  → categórica
data_csv['OverallQual'].dtype     # int64       → numérica discreta (1..10)
data_csv['SalePrice'].dtype       # int64/float → continua
data_csv['LotFrontage'].dtype     # float64     → continua (con nulos)
```

Atajos para separarlas de golpe:

```python
data_csv.select_dtypes(include='number').columns   # todas numéricas
data_csv.select_dtypes(include='object').columns   # todas categóricas (texto)
```

Regla práctica para clasificar:

| dtype | # únicos | Interpretación |
|---|---|---|
| `object`/`str` | — | categórica (mira `value_counts` para ver si tiene orden natural, p.ej. `Po < Fa < TA < Gd < Ex`) |
| `int64` | pocos (≤ ~15) | discreta (p.ej. `OverallQual`, `FullBath`) |
| `int64`/`float64` | muchos | continua (p.ej. `SalePrice`, `GrLivArea`) |

---

## 3. Ver si hay nulos

```python
data_csv['LotFrontage'].isnull().sum()     # número de nulos en esa columna
data_csv['LotFrontage'].isna().sum()       # isna() == isnull(), idénticos
data_csv.isnull().sum()                    # nulos por columna (todas a la vez)
data_csv.isnull().mean() * 100             # % de nulos por columna
```

También `data_csv.info()` (que ya usaste) te da `Non-Null Count` por columna de un vistazo.

---

## 4. Tratamiento de nulos (si hay)

Depende del caso. Las funciones típicas:

```python
# a) Rellenar con un valor constante (categóricas: "None" para Alley, PoolQC, etc.)
data_csv['Alley'] = data_csv['Alley'].fillna('NoAlley')

# b) Rellenar con la media/mediana (numéricas continuas)
data_csv['LotFrontage'] = data_csv['LotFrontage'].fillna(
    data_csv['LotFrontage'].median()
)

# c) Rellenar con la mediana agrupada por vecindario (más fino)
data_csv['LotFrontage'] = data_csv['LotFrontage'].fillna(
    data_csv.groupby('Neighborhood')['LotFrontage'].transform('median')
)

# d) Interpolar (series ordenadas)
data_csv['LotFrontage'] = data_csv['LotFrontage'].interpolate()

# e) Borrar filas con nulos en esa columna
data_csv = data_csv.dropna(subset=['LotFrontage'])

# f) Borrar la columna entera si tiene casi todo nulo (p.ej. PoolQC ~99%)
data_csv = data_csv.drop(columns=['PoolQC'])
```

En el enunciado te piden **"el tratamiento que se te ocurre"** → basta con justificar la elección (p.ej. *"Alley tiene 91 no nulos de 1460; los NA significan 'sin callejón', así que los relleno con 'NoAlley'"*).

---

## 5. Numéricas: mínimo, máximo y outliers

```python
data_csv['SalePrice'].min()
data_csv['SalePrice'].max()
data_csv['SalePrice'].describe()     # min, 25%, 50%, 75%, max, media, std de golpe
```

Para **outliers**, el método estándar es el **IQR**:

```python
col = 'SalePrice'
Q1 = data_csv[col].quantile(0.25)
Q3 = data_csv[col].quantile(0.75)
IQR = Q3 - Q1
low  = Q1 - 1.5 * IQR
high = Q3 + 1.5 * IQR

outliers = data_csv[(data_csv[col] < low) | (data_csv[col] > high)]
print(len(outliers))
```

Equivalentemente en una línea (útil para muchas columnas):

```python
numericas = data_csv.select_dtypes('number').columns
for c in numericas:
    Q1, Q3 = data_csv[c].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    n_out = ((data_csv[c] < Q1 - 1.5*IQR) | (data_csv[c] > Q3 + 1.5*IQR)).sum()
    print(c, n_out)
```

Visualmente (aquí sí entra seaborn):

```python
import seaborn as sns
sns.boxplot(x=data_csv['SalePrice'])
```

> Nota: `data_csv[col].max()` es preferible a `max(data_csv[col])` porque:
> - es más rápido,
> - ignora `NaN` por defecto,
> - mantiene el nombre del índice (útil luego para `idxmax()`, que te da la **fila** del máximo).

Si quieres la **fila** del máximo: `data_csv.loc[data_csv['SalePrice'].idxmax()]`.

---

## 6. Decidir si eliminar una columna

No hay función que "decida": es criterio tuyo + comprobaciones:

```python
data_csv['Id'].nunique()              # si == nº filas → es solo un identificador
data_csv['Utilities'].value_counts()  # si una categoría domina al 99% → poca info
```

Para borrarla:

```python
data_csv = data_csv.drop(columns=['Id'])
# o también:
data_csv.drop(columns=['Id'], inplace=True)
```

Criterios típicos para eliminar:
- **ID**: no aporta al modelo (es único por fila) → eliminar.
- **Varianza casi nula** (`Utilities` casi todo "AllPub") → eliminar.
- **Casi todos nulos** (`PoolQC`, `MiscFeature`) → valorar eliminar o imputar como "No".

---

## 7. Recodificación de categóricas

Depende de si la categoría **tiene orden** o no:

```python
# a) Categórica ORDENADA → map() o pd.Categorical con orden
qual_map = {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
data_csv['ExterQual_num'] = data_csv['ExterQual'].map(qual_map)

# o preservando el orden como categoría de pandas:
data_csv['ExterQual'] = pd.Categorical(
    data_csv['ExterQual'],
    categories=['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    ordered=True
)
```

```python
# b) Categórica SIN orden → one-hot encoding con get_dummies
data_csv = pd.get_dummies(data_csv, columns=['MSZoning', 'Street'],
                          drop_first=True)   # drop_first evita colinealidad

# c) Sustitución simple de valores
data_csv['CentralAir'] = data_csv['CentralAir'].replace({'Y': 1, 'N': 0})
```

Opcionalmente con `sklearn` (no necesario para esta práctica):

```python
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
```

**Justificación típica para el informe**: *"ExterQual es cualitativa ordinal (Po<Fa<TA<Gd<Ex), la convierto a 1-5 con `map` para que el modelo entienda el orden. MSZoning no tiene orden, uso `get_dummies` con `drop_first=True`."*

---

## Resumen: ¿pandas o Python builtin?

| Tarea | Herramienta recomendada |
|---|---|
| Únicos / frecuencias | `Series.unique()`, `.nunique()`, `.value_counts()` |
| Tipo | `Series.dtype` + criterio humano (o `select_dtypes`) |
| Nulos | `.isnull().sum()`, `info()` |
| Imputar | `.fillna()`, `.interpolate()`, `groupby().transform()` |
| Min/max/estadísticos | `.min()`, `.max()`, `.describe()`, `.quantile()` |
| Outliers | IQR con `quantile` (+ `sns.boxplot` para verlo) |
| Eliminar columna | `.drop(columns=[...])` |
| Recodificar | `.map()`, `.replace()`, `pd.Categorical()`, `pd.get_dummies()` |

**No necesitas `max()`/`min()` de Python** — los métodos `.max()`/`.min()` de pandas hacen lo mismo, mejor (ignoran `NaN`, son vectorizados y permiten encadenar con `idxmax`, `describe`, etc.). Reserva los builtins para cosas tipo `len()`, `print()`, `sorted()` sobre listas pequeñas.

¿Quieres que te monte un **bloque de código reutilizable** que, dada una columna, imprima automáticamente todo este informe (únicos, tipo, nulos, min/max, outliers)? Te lo puedo preparar como función para que la llames columna a columna y te ahorres repetir código.
